# CH5 - Exercise 6

We continue to consider the use of a logistic regression model to
predict the probability of `default` using `income` and `balance` on the
`Default` data set. In particular, we will now compute estimates for the
standard errors of the `income` and `balance` logistic regression coefficients
in two different ways: (1) using the bootstrap, and (2) using the
standard formula for computing the standard errors in the `sm.GLM()`
function. Do not forget to set a random seed before beginning your
analysis.

**(a)** Using the `summarize()` and `sm.GLM()` functions, determine the
estimated standard errors for the coefficients associated with
`income` and `balance` in a multiple logistic regression model that
uses both predictors.

**(b)** Write a function, `boot_fn()`, that takes as input the `Default` data
set as well as an index of the observations, and that outputs
the coefficient estimates for `income` and `balance` in the multiple
logistic regression model.

**(c)** Following the bootstrap example in the lab, use your `boot_fn()`
function to estimate the standard errors of the logistic regression
coefficients for `income` and `balance`.

**(d)** Comment on the estimated standard errors obtained using the
`sm.GLM()` function and using the bootstrap.

In [1]:
import numpy as np
import pandas as pd
from matplotlib .pyplot import subplots
import statsmodels .api as sm
from ISLP import load_data
from ISLP.models import ( ModelSpec as MS ,
summarize )

from ISLP import confusion_table
from ISLP.models import contrast
from sklearn. discriminant_analysis import \
( LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn. naive_bayes import GaussianNB
from sklearn. neighbors import KNeighborsClassifier
from sklearn. preprocessing import StandardScaler
from sklearn. model_selection import train_test_split
from sklearn. linear_model import LogisticRegression

from functools import partial
from sklearn.model_selection import (
    train_test_split, cross_validate, KFold, ShuffleSplit
)
from sklearn import clone 
from ISLP.models import sklearn_sm


In [2]:
Default = load_data("Default")
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


### Model fitted over the whole dataset  

In [21]:
design  = MS(['balance', 'income'])
X = design.fit_transform(Default)
y = (Default['default'] == 'Yes')

model_tot   = sm.GLM(y, X, family=sm.families.Binomial())
results_tot = model_tot.fit()
summarize(results_tot)



,coef,std err,z,P>|z|
intercept,-11.540500,0.435000,-26.544,0.0
balance,0.005600,0.000000,24.835,0.0
income,0.000021,0.000005,4.174,0.0


In [16]:
def boot_fn(data,idx):
    design  = MS(['balance', 'income'])
    X = design.fit_transform(data.loc[idx])
    y = (data['default'].loc[idx] == 'Yes')

    model_idx  = sm.GLM(y, X, family=sm.families.Binomial())
    results_idx = model_idx.fit()
    return results_idx.params

# test on the whole dataset
boot_fn(Default,range(Default['balance'].shape[0]))

intercept   -11.540468
balance       0.005647
income        0.000021
dtype: float64

Define a function to compute the SE for $\beta_1$ and $\beta_2$ 

In [19]:
def boot_SE(func,
            D,
            n=None,
            B=1000,
            seed=0):
    rng = np.random.default_rng(seed)

    first_1, second_1,first_2, second_2 = 0,0,0,0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,
                        n,
                        replace = True)
        
        _,beta1,beta2 = func(D,idx)
        first_1 += beta1
        first_2 += beta2
        second_1 += beta1**2
        second_2 += beta2**2
    SE_1 = np.sqrt(second_1/B - (first_1/B)**2)
    SE_2 = np.sqrt(second_2/B - (first_2/B)**2)
    
    return SE_1,SE_2


In [20]:
boot_SE(func=boot_fn,
            D=Default,
            n=None,
            B=1000,
            seed=0)

(np.float64(0.00023043534317828486), np.float64(4.767199749821317e-06))